# L17 · 用户鉴权：守住你的服务大门

**学习目标**
- 理解「鉴权」：证明「你是你」才能进门
- 理解 Token（令牌）的工作原理
- 用密码哈希 + Token 实现一个最小可用登录

**前置依赖**：L16（数据库）  
**预计时长**：45 分钟  
**技术栈**：`fastapi`、`uvicorn`、`requests`、`sqlite3`、`hashlib`（标准库）

---

## 概念讲解：鉴权 = 门口的保安查「通行证」

开放 API 谁都能调，很危险。所以需要「登录拿令牌（Token）」：
1. 你用账号密码登录 → 服务器验证正确 → 发你一张「临时通行证」Token
2. 之后每次请求，在请求头带 `Authorization: Bearer <Token>`
3. 服务器验票通过才放行

**密码绝不存明文**，要用哈希（hash）变成不可逆的乱码——即使数据库泄露，密码也看不出。

## 第一步：密码哈希（不可逆加密）

In [ ]:
import hashlib, secrets

def hash_pw(password: str, salt: str = None):
    if salt is None:
        salt = secrets.token_hex(8)            # 随机盐，防彩虹表
    h = hashlib.sha256((password + salt).encode()).hexdigest()
    return salt, h

salt, h = hash_pw("123456")
print("盐：", salt)
print("哈希后：", h, "（明文 123456 已看不见）")

## 第二步：登录发 Token + 受保护接口

In [ ]:
from fastapi import FastAPI, HTTPException, Header
import sqlite3, uvicorn, threading, time, requests, json

# 简单内存用户库（生产用 DB）：用户名 -> (salt, hash)
users = {}
s, h = hash_pw("123456"); users["alice"] = (s, h)
valid_tokens = set()
app = FastAPI()

@app.post("/login")
def login(username: str, password: str):
    if username not in users:
        raise HTTPException(401, "用户不存在")
    salt, true_h = users[username]
    _, try_h = hash_pw(password, salt)
    if try_h != true_h:
        raise HTTPException(401, "密码错误")
    token = secrets.token_hex(16)
    valid_tokens.add(token)
    return {"token": token}

@app.get("/secret")
def secret(authorization: str = Header(None)):
    token = (authorization or "").replace("Bearer ", "")
    if token not in valid_tokens:
        raise HTTPException(403, "无有效令牌，禁止访问")
    return {"msg": "🎉 欢迎进入机密区，这是只有登录用户才能看到的数据"}

PORT = 8774
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning"), daemon=True).start()
time.sleep(2)
print("✅ 带鉴权的服务已上线（账号 alice / 密码 123456）")

# 🎯 AHA 顿悟单元格：没有令牌，门就是关的

运行下面代码。你会看到三段剧情：
1. **没令牌直接访问** → 被 403 拒之门外；
2. **用正确账号密码登录** → 拿到 Token；
3. **带 Token 再访问** → 顺利进入机密区。

> 你刚刚实现了和微信/银行 App 同源的「登录-发令牌-验票」机制。安全，从这一课开始。

In [ ]:
# ===== 运行我！（需先运行上面服务）=====
import requests, json
BASE = f"http://127.0.0.1:{PORT}"

print("  ① 没令牌，硬闯机密区：")
r = requests.get(f"{BASE}/secret")
print("   状态码", r.status_code, "-", r.json()["detail"])

print("\n  ② 用正确账号登录：")
login = requests.post(f"{BASE}/login", data={"username": "alice", "password": "123456"})
token = login.json()["token"]
print("   拿到 Token 前 12 位：", token[:12], "...")

print("\n  ③ 带 Token 再访问：")
r2 = requests.get(f"{BASE}/secret", headers={"Authorization": f"Bearer {token}"})
print("   状态码", r2.status_code, "-", r2.json()["msg"])
print("\n  🔐 你亲手建起了一道和银行同源的鉴权大门！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：哈希不可逆；Token 作为「临时票」的心智；Header 传参。  
**易错点**：HTTP Basic Auth 表单需 `data=` 而非 `json=`；Header 名大小写不敏感但 FastAPI 用 `Header`。  
**AHA 机制**：403→登录→200 三幕剧，强「安全门」实感。  
**安全红线**：明确告知学员这是教学最简版，生产要用 JWT+HTTPS+速率限制，绝不存明文密码。  
**衔接**：L18 部署；L25-L30 工程化约束/安全对齐会深化。  
**依赖**：标准库 hashlib/secrets + fastapi/uvicorn/requests。

# 📚 作业 / 下一步

1. 用错误密码登录，看返回 401。
2. 把 `valid_tokens` 改成存到数据库，实现「多设备登录」。
3. 下一课 **L18 部署初探：让你的服务上线** —— 把本地服务变成「别人也能访问」的真服务。